In [ ]:
# full aggregated flight data
#this piece takes the monthly raw data and puts it all 
# together into one csv for the year

from pathlib import Path
import pandas as pd

# Folder containing your 12 csv files
DATA_DIR = Path.home() / "Downloads"/"2023_flight_data"

# Find all csv files 
csv_files = sorted(DATA_DIR.glob("*.csv"))

# Read and combine csv files
df_list = []

for file in csv_files:
    print(f"Reading {file.name}")
    df = pd.read_csv(file)
    df_list.append(df)

# Combine into one DataFrame
full_year_df = pd.concat(df_list, ignore_index=True)

# Save result
#output_file = DATA_DIR / "2025_full_combined_flight_data.csv"
output_file = DATA_DIR / "2023_full_combined_flight_data.csv"
full_year_df.to_csv(output_file, index=False)

print(f"Saved combined file to: {output_file}")

In [1]:
#pull just the 50 airports
#from the large dataset made in above code chunk, this pulls 50 airports we wanted. 

from pathlib import Path
import pandas as pd

INPUT_FILE = #path to large csv file with data for the year
OUTPUT_FILE = #output path to where you want to keep new file with pulled 50 airports 

AIRPORTS = [
    "TPA","STL","SNA","SMF","SLC","SJC","SFO","SEA","SAT","SAN","RSW","RDU","PIT",
    "PHX","PHL","PDX","ORD","OGG","OAK","MSY","MSP","MIA","MDW","MCO","MCI","LGA",
    "LAX","LAS","JFK","IND","IAH","IAD","HOU","HNL","FLL","EWR","DTW","DFW","DEN",
    "DCA","DAL","CVG","CMH","CLT","CLE","BWI","BOS","BNA","AUS","ATL"
]

airport_set = set(AIRPORTS)

#filter in chunks (because datset was so large)
chunksize = 500_000
first_write = True

for chunk in pd.read_csv(INPUT_FILE, chunksize=chunksize):
    
    filtered_chunk = chunk[
        chunk["Origin"].isin(airport_set) |
        chunk["Dest"].isin(airport_set)
    ]
    
    # write incrementally (avoids memory overload)
    filtered_chunk.to_csv(
        OUTPUT_FILE,
        mode="w" if first_write else "a",
        header=first_write,
        index=False
    )
    
    first_write = False
    print(f"Processed chunk, kept {len(filtered_chunk)} rows")

print("Done. Filtered file saved.")

Done. Filtered file saved.


In [ ]:
#this organizes by hour dataset that's got just the 50 airports by hour

from pathlib import Path
import pandas as pd

INPUT_FILE = #path to large csv file with data for the year
OUTPUT_FILE = #output path to where you want to keep new file with pulled 50 airports 

AIRPORTS = [
    "TPA","STL","SNA","SMF","SLC","SJC","SFO","SEA","SAT","SAN","RSW","RDU","PIT",
    "PHX","PHL","PDX","ORD","OGG","OAK","MSY","MSP","MIA","MDW","MCO","MCI","LGA",
    "LAX","LAS","JFK","IND","IAH","IAD","HOU","HNL","FLL","EWR","DTW","DFW","DEN",
    "DCA","DAL","CVG","CMH","CLT","CLE","BWI","BOS","BNA","AUS","ATL"
]

airport_set = set(AIRPORTS)

#set chunk size
chunksize = 500_000

#set variable to true for writing to csv
first = True

all_hours = list(range(24))

#process chunks in large dataset to organize by hour
for chunk in pd.read_csv(INPUT_FILE, chunksize=chunksize, low_memory=False):

    chunk.columns = chunk.columns.str.strip()

    chunk["FlightDate"] = pd.to_datetime(chunk["FlightDate"], errors="coerce")
    chunk["DepTime"] = pd.to_numeric(chunk["DepTime"], errors="coerce")
    chunk["ArrTime"] = pd.to_numeric(chunk["ArrTime"], errors="coerce")

    chunk = chunk.dropna(subset=["FlightDate", "DepTime", "ArrTime"])

    chunk["DepHour"] = (chunk["DepTime"] // 100).astype(int)
    chunk["ArrHour"] = (chunk["ArrTime"] // 100).astype(int)

    #departures
    dep = chunk[chunk["Origin"].isin(airport_set)].groupby(
        ["FlightDate", "Origin", "DepHour"]
    ).size().reset_index(name="Departures")

    dep.rename(columns={"Origin": "Airport", "DepHour": "Hour"}, inplace=True)

    # arrivals
    arr = chunk[chunk["Dest"].isin(airport_set)].groupby(
        ["FlightDate", "Dest", "ArrHour"]
    ).size().reset_index(name="Arrivals")

    arr.rename(columns={"Dest": "Airport", "ArrHour": "Hour"}, inplace=True)

    #merge
    merged = pd.merge(dep, arr, on=["FlightDate", "Airport", "Hour"], how="outer")

    # build grid of dates
    dates = merged["FlightDate"].dropna().unique()
    airports = AIRPORTS

    full_index = pd.MultiIndex.from_product(
        [dates, airports, all_hours],
        names=["FlightDate", "Airport", "Hour"]
    )

    merged = merged.set_index(["FlightDate", "Airport", "Hour"]).reindex(full_index).reset_index()

    # fill in columns
    #any NaN values will be changed to 0
    merged["Departures"] = merged["Departures"].fillna(0).astype(int)
    merged["Arrivals"] = merged["Arrivals"].fillna(0).astype(int)

    #create total flights column that adds departures and arrivals
    merged["TotalFlights"] = merged["Departures"] + merged["Arrivals"]

    # format hour
    merged["Hour"] = merged["Hour"].apply(lambda x: f"{int(x):02d}:00")

    # write chunk to csv
    merged.to_csv(
        OUTPUT_FILE,
        mode="w" if first else "a",
        header=first,
        index=False
    )

    first = False
    print(f"Processed chunk → {len(merged)} rows")

print("Done.")

Processed chunk → 37200 rows
Processed chunk → 58800 rows
Processed chunk → 73200 rows
Processed chunk → 73200 rows
Processed chunk → 70800 rows
Processed chunk → 70800 rows
Processed chunk → 73200 rows
Processed chunk → 73200 rows
Processed chunk → 73200 rows
Processed chunk → 73200 rows
Processed chunk → 37200 rows
Processed chunk → 74400 rows
Processed chunk → 73200 rows
Processed chunk → 36000 rows
Done.


In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.home()

# INPUT = DATA_DIR / "2023_hourly_airport_demand_2023.csv"
# OUTPUT = DATA_DIR / "2023_hourly_airport_demand_2023_CLEAN.csv"
INPUT = DATA_DIR / "2025_hourly_total_flights_data.csv"
OUTPUT = DATA_DIR / "2025_hourly_total_flights_data_CLEAN.csv"

df = pd.read_csv(INPUT)

# ensure consistent types
df["FlightDate"] = pd.to_datetime(df["FlightDate"])
df["Hour"] = df["Hour"].astype(str).str.strip()
df["Airport"] = df["Airport"].astype(str).str.strip()

# collapse duplicates
df_clean = df.groupby(
    ["FlightDate", "Airport", "Hour"],
    as_index=False
).agg({
    "Departures": "sum",
    "Arrivals": "sum",
    "TotalFlights": "sum"
})

#sort for readability
df_clean = df_clean.sort_values(["FlightDate", "Airport", "Hour"])

# save to csv
df_clean.to_csv(OUTPUT, index=False)
print(df_clean.duplicated(["FlightDate", "Airport", "Hour"]).sum())

print("Done. Clean file saved.")

0
Done. Clean file saved.
